03/16　テスト用にめちゃくちゃ簡単な設計をやりたい

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

In [2]:
class RPGEnv:

    def __init__(self):
        self.reset()

    def reset(self):
        self.player_hp = 100
        self.boss_hp = 100
        self.turn = 0
        return self._get_state()

    def _get_state(self):
        return [self.player_hp, self.boss_hp, self.turn]

    def step(self, action):
        self.turn += 1

        if action == 0:
            self.player_hp -= 10
        elif action == 1:
            pass
        elif action == 2:
            self.boss_hp += 5

        # プレイヤー攻撃（固定）
        self.boss_hp -= 10

        done = False
        reward = 0

        if self.boss_hp <= 0:
            done = True
            if self.turn <= 30:
                reward = 1

        if self.player_hp <= 0:
            done = True

        return self._get_state(), reward, done

In [3]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size, 32),
            nn.ReLU(),
            nn.Linear(32, action_size)
        )

    def forward(self, x):
        return self.fc(x)

In [4]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.q_net = QNetwork(state_size, action_size)
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=0.001)

        self.gamma = 0.99
        self.epsilon = 0.1

        self.action_size = action_size

    def get_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_size)

        state = torch.FloatTensor(state).unsqueeze(0)
        q_values = self.q_net(state)
        return torch.argmax(q_values).item()

    def train(self, state, action, reward, next_state, done):
        state = torch.FloatTensor(state)
        next_state = torch.FloatTensor(next_state)

        q_values = self.q_net(state)
        q_value = q_values[action]

        with torch.no_grad():
            next_q_values = self.q_net(next_state)
            max_next_q = torch.max(next_q_values)

            target = reward
            if not done:
                target += self.gamma * max_next_q

        loss = (q_value - target) ** 2

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

In [5]:
env = RPGEnv()
agent = DQNAgent(state_size=3, action_size=3)

episodes = 500

for episode in range(episodes):
    state = env.reset()
    total_reward = 0

    while True:
        action = agent.get_action(state)
        next_state, reward, done = env.step(action)

        agent.train(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

        if done:
            break

    if episode % 50 == 0:
        print(f"Episode {episode}, Reward: {total_reward}")

Episode 0, Reward: 1
Episode 50, Reward: 1
Episode 100, Reward: 1
Episode 150, Reward: 1
Episode 200, Reward: 1
Episode 250, Reward: 1
Episode 300, Reward: 1
Episode 350, Reward: 1
Episode 400, Reward: 1
Episode 450, Reward: 1
